# Convert each transaction with the rate valid at the time

**Track:** pyspark | **Difficulty:** hard | **Tags:** as-of join, point-in-time join, range join, window functions

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("010-asof-exchange-rate-join")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "exchange_rates": """
UEFSMRUEFUYVRkwVChUAEgAAIxgDAAAAR0JQAQcIRVVSAQdAVVNEAwAAAEFFRAMAAABTR0QVABVaFV4sFbQBFRAVBhUGHDYAKANV
U0QYA0FFRBERAAAALbADAAAAtAEBAxmIBmzioGmDSAZYpo3RhjAhhQ0TBAnMEI5CGGSZpkSJRDEQAAAVBBXmBhXoBkwVrgEVABIA
ALMD8EAAABAdqAAADdDYAAAMto0AAAN2JgAAEE1nAAAQNqcAAAN39wAAA3SEAAAMwE4AAAmPFgAAA3+0AAAP/5cAAAygfwEjeEoA
AAy4qQAAA39fAAADeXcAABBCxgAADMCrAAAJkYQBBQCVAQXwPKsAAA3aGAAAEASfAAAQJP4AAANw5wAADbi7AAADck4AAAzSzAAA
A3ZqAAADfC0AAAlqmAAADbvIAAAMq1oBIwDoARnw/+gAABAj3AAADc2pAAAJgWkAAA3JmgAADdfUAAAJdIgAAAlkMQAADLkWAAAQ
Ki4AAAN9GwAAA4ClAAAQAqQAAAyxiQAAEBZNAAAMuawAABBM2gAADKg6AAAM1pkAABAa0QAACYx6AAANsV0AAAN+FAAAEEzCAAAN
z2IAAAmF0gAAA3FuAAAJf0gAAAzAtQAAECreAAANt60AAAlzjAAADbgvAAAQHgkAAA3I+QAAA3ffAAAN3EEAAAN7nAAADJ3uAAAM
0zIAAA3EagAADdbjAAAN5pwAAA20tQAADKo4AAAM0xsAAAmDfgAADN3JAAAJgH4AAA3ydwAAED/aAAAMoOoVABW6ARXAASwVtAEV
EBUGFQYcGAUAABBNZxgFAAADcOcWACgFAAAQTWcYBQAAA3DnEREAAABd8FwDAAAAtAEBBxmAgGBAKBgOiIRiwWg4HpCIZEKpWC6Y
jGbD6Xg+oJBoRCqZTqiUasVquV4HWEw2o9VsN1xOt+P1fD9gUDgkFo1HZLKhXDJlTecTGpVOqVUrAAAAAAAVBBXAChWQCUwVqAEV
ABIAAKAFUABzUIjWMQYAAOcU9z0yBgAAHB1m+wEQDAWvgbABCAxPk5UiARgMLqE6CQEIDCDEN6wBGAxwhfA7ARAM33rrrwEQDGRS
OgQBEAxjNTAOAQgMbsRITAEIDNoFB/ABIAzIcwvbAQgMEbSBUwEYDKUrTscBEAymz+vAAQgMH5XFLwEYDKjE7k0BCAxjSEpZAQgM
IZc9/wEgDFOha0kBEAxkbogSAQgMpWPq4wEYDM/5QDYBEAx1Tr8mAQgMf4gxUgEIDEKJmBgBCAw31f0IAQgIeKXdBQgMyh1YAAEQ
DPNQ4TgBCAyAhPr1AUgInFBoBUgMrTCfSgEYDIbtJVsBCAzKu0bOASAIpKP+BcAMLhFzQgEYDFRJSsIBGAhKoFElIAyJMSryARAM
YERk3QEIDAo3gLsBCAiZNSclAAwu4grpARAMeJypRQFACH4lZAWADCGumMkBGAxF28O2AQgIMxYfBRAMbs18DwEoDI0ViwMBCAzf
dqowAQgI8AsrBWAMyN5Q0AEwDL63+O8BCAwyA/taASAMurw8FAEIDOsGf1YBCAy3gWxAAQgM+5pDGwEIDCQvLMEBMAyIAQaxAQgM
f7bn5gEIDDt8z7kBCAhaiAAlYAg28I8lkAxWULXVARgM4NSEBgFADEFjWl8BCAwvEi8qAQgIlOq3BdAMnVEkHgEQDG3V9OoBMAxk
H5ErARAMt/4ZvAEQDEd36dQBCAzOm2ZgARgMcroR5AEQDC6YBkYBEAygPf0KAQg4l9/E9TEGAAC3NrbYMQYAFQAVugEVwAEsFbQB
FRAVBhUGHBgIAM6bZmAyBgAYCAAgxDesMQYAFgAoCADOm2ZgMgYAGAgAIMQ3rDEGABERAAAAXfBcAwAAALQBAQcZgIBgQCgYDoiE
YsFoOB6QiGRCqVgumIxmw+l4PiBDSDQilUwnVEq1YrVchxcsZpDNaDXbDZfT7Xg93w8YFA6JReOxgUwaYsols+l8QqPSKQAAAAAA
FQQZTDUAGAZzY2hlbWEVBgAVDCUCGAhjdXJyZW5jeSUATBwAAAAVDhUKFQIYC3JhdGVfdG9faW5yJQoVCBUULFwVCBUUAAAAFQQl
AhgKdmFsaWRfZnJvbSUUTIwSHCwAAAAAABa0ARkcGTwmABwVDBk1AAYQGRgIY3VycmVuY3kVAha0ARaAAhaEAiZqJggcNgAoA1VT
RBgDQUVEEREAGSwVBBUAFQIAFQAVEBUCADwWnAQZBhkmALQBAAAAJgAcFQ4ZNQAGEBkYC3JhdGVfdG9faW5yFQIWtAEWrgkWtgkm
lgkmjAIcGAUAABBNZxgFAAADcOcWACgFAAAQTWcYBQAAA3DnEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmALQBAAAAJgAcFQQZNQAG
EBkYCnZhbGlkX2Zyb20VAha0ARagDRb2Cyb0FCbCCxwYCADOm2ZgMgYAGAgAIMQ3rDEGABYAKAgAzptmYDIGABgIACDEN6wxBgAR
EQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAtAEAAAAWzhgWtAEmCBawFwAZHBgMQVJST1c6c2NoZW1hGMwCLy8vLy8vQUFBQUFRQUFB
QUFBQUtBQXdBQmdBRkFBZ0FDZ0FBQUFBQkJBQU1BQUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFBQU1BQUFDUUFBQUFRQUFBQUFRQUFB
Q00vLy8vQUFBQkNoQUFBQUFrQUFBQUJBQUFBQUFBQUFBS0FBQUFkbUZzYVdSZlpuSnZiUUFBQUFBR0FBZ0FCZ0FHQUFBQUFBQUNB
TVQvLy84QUFBRUhFQUFBQUNRQUFBQUVBQUFBQUFBQUFBc0FBQUJ5WVhSbFgzUnZYMmx1Y2dBSUFBd0FCQUFJQUFnQUFBQUtBQUFB
QkFBQUFCQUFGQUFJQUFZQUJ3QU1BQUFBRUFBUUFBQUFBQUFCQlJBQUFBQWdBQUFBQkFBQUFBQUFBQUFJQUFBQVkzVnljbVZ1WTNr
QUFBQUFCQUFFQUFRQUFBQT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRk8HAAAHAAAHAAAAC4DAABQQVIx
""",
    "transactions": """
UEFSMRUEFYAUFZoKTBXAAhUAEgAAgAoIMXUABQEAMg0IADMNCAA0DQgANQ0IADYNCAA3DQgAOA0IADkNCAA6DQgAOw0IADwNCAA9
DQgAPg0IAD8NCABADQgAQQ0IAEINCABDDQgARA0IAEUNCABGDQgARw0IAEgNCABJDQgASg0IAEsNCABMDQgATQ0IAE4NCABPDQgA
UA0IAFENCABSDQgAUw0IAFQNCABVDQgAVg0IAFcNCABYDQgAWQ0IAFoNCABbDQgAXA0IAF0NCABeDQgAXw0IAGANCABhDQgAYg0I
AGMNCABkDQgAZQ0IAGYNCABnDQgAaA0IAGkNCABqDQgAaw0IAGwNCABtDQgAbg0IAG8NCABwDQgAcQ0IAHINCABzDQgAdA0IAHUN
CAB2DQgAdw0IAHgNCAB5DQgAeg0IAHsNCAB8DQgAfQ0IAH4NCAB/DQgAgA0IAIENCACCDQgAgw0IAIQNCACFDQgAhg0IAIcNCACI
DQgAiQ0IAIoNCACLDQgAjA0IAI0NCACODQgAjw0IAJANCACRDQgAkg0IAJMNCACUDQgAlQ0IAJYNCACXDQgAmA0IAJkNCACaDQgA
mw0IAJwNCACdDQgAng0IAJ8NCACgDQgAoQ0IAKINCACjDQgApA0IAKUNCACmDQgApw0IAKgNCACpDQgAqg0IAKsNCACsDQgArQ0I
AK4NCACvDQgAsA0IALENCACyDQgAsw0IALQNCAC1DQgAtg0IALcNCAC4DQgAuQ0IALoNCAC7DQgAvA0IAL0NCAC+DQgAvw0IAMAN
CADBDQgAwg0IAMMNCADEDQgAxQ0IAMYNCADHDQgAyA0IAMkNCADKDQgAyw0IAMwNCADNDQgAzg0IPM91AAAAAAAA0HUAAAAAAAAV
ABXSAhXaAiwVwAIVEBUGFQYcGAjQdQAAAAAAABgIMXUAAAAAAAAWACgI0HUAAAAAAAAYCDF1AAAAAAAAEREAAACpAfCoAwAAAMAC
AQgpAAECAwQFBgcICQoLDA0ODxAREhMUFRYXGBkaGxwdHh8gISIjJCUmJygpKissLS4vMDEyMzQ1Njc4OTo7PD0+P0BBQkNERUZH
SElKS0xNTk9QUVJTVFVWV1hZWltcXV5fYGFiY2RlZmdoaWprbG1ub3BxcnN0dXZ3eHl6e3x9fn+AgYKDhIWGh4iJiouMjY6PkJGS
k5SVlpeYmZqbnJ2enxUEFWIVXEwVDhUAEgAAMRgDAAAAVVNEAQcEU0cFBwhKUFkBDghHQlABB0BFVVIDAAAASU5SAwAAAEFFRBUA
FYoBFZABLBXAAhUQFQYVBhw2ACgDVVNEGANBRUQREQAAAEXwRAMAAADAAgEDKUAAZQSxGtzSFdk2IMGKGMCYDGMADiyAoESJAGPB
MYwXICADBJsLDAWACIhhsGyAhAmSjKXRBQAABCMidhUEFYAPFdwMTBXAAhUAEgAAwAd0AAAAB2snAAAAAOcSAAAABub1AAAABPZy
AAAAB5k/AQwE3GUBDBxnTQAAAAKJewEMBAmTAQYEhOEBHhxkIAAAAAMRZgE8HAy8AAAABT1LARgEbYYBGAQhiQEYBOwlATAEXNsB
GAThHwFIBG2CAR4EypoBGBhphwAAAAEDARIIAO5ZAUIEwUABNgSyIwEGBDyUASoEmLYBNgTF6wFCBMyXATAEitcBtAB6AWYIBq1s
AQwEaY4BJAQEPwE2BOaoAQwEC58BTgQr8AEeAM4BQgQFvwESCAIgygFUBDqFARgEAsgBKgSd+QESAG0BogQHNQFCCATbeAESBDIC
AaIETwMBBgR2MAESBHJdATAEQv0BDARCSgEYAIUBugQGeAFICAcGIAGcALABKgQCHgGoCACTWAEkAEAF0gTRgAE2BM9UAUIE/7oB
GAR0QwHSBPkRAbQE7KoBBgRJ5QEkBGRFAZwEeJUBEgTBxQESACwBrggDFtUBYAS4tQESBIMuAR4EfmkB5ATc5gEGAEQBZggBBvYB
JBx5vAAAAAN0vgEkBD+hAU4AcAFaCAeVTgEMBEu9ATAAwwGcCAXLYQGKBAqyAZYETY0BMASKUgE8BEvjARIESZsBDAR+/wE2BMMh
ARIErRMBBgRfdAFCBBT3ARgA+QGiIAJ7XgAAAAQwnAEeBD2ZAQwAxAHACAWB8gGQBLZ+AQYEht0BHgQ4mAEMBIpUAVoArAHYBAFH
IVAEBKMh7AgHh9EBPASlwwFaBDJqAWYAtkGaBARYAWAIA2SKARIAVyEICAPJ+wFIBKRGATAE/iwBWgSENwE2BPd7ARIEbvQBKgD+
AbQIBLbPAR4APAEqCAGwaAF4AIoBQgQAkyF6CAJlKwH2BMqwASQE6+AhDgQ9MgFCABUBTggCGnABZgC3IXQIAuzCAVQEXewBBgRh
kQEwBHBJAXIEdnEBBgSPswEGBCnXAR4ALQHeBAFTAX4EAHkh7AgBETkBVASuWQEGBLcUASQEWbwBMADfIbAIAoVmAQwAhQEYBAIB
RcoAcgHeBASnARcIBkZHAaIESy8BDTQ0twAAAAPF7QAAAAatdhUAFdICFdoCLBXAAhUQFQYVBhwYBgAAAAeZPxgGAAAAAALIFgAo
BgAAAAeZPxgGAAAAAALIEREAAACpAfCoAwAAAMACAQgpAAECAwQFBgcICQoLDA0ODxAREhMUFRYXGBkaGxwdHh8gISIjJCUmJygp
KissLS4vMDEyMzQ1Njc4OTo7PD0+P0BBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZWltcXV5fYGFiY2RlZmdoaWprbG1ub3BxcnN0
dXZ3eHl6e3x9fn+AgYKDhIWGh4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2enxUEFYAUFdQSTBXAAhUAEgAAgAqQgHiM2bQxBgBAiOAv
OjIGAIDLxdbBMQYAgI2kZEIyBgAAzk6LGgEICAjPtQEgMEA0QAndMQYAQPz/fPoBCAwv6Y7MAQgMAf40UwEo8FtkbogSMgYAwLEV
dVwyBgCAZZX05DEGAICvDJVGMgYAwGk5YwsyBgCAFQrFDTIGAMD5j2MBMgYAAAtzpiUyBgDARpA9TTIGAECDVN41MgYAQK6Mkuwx
BgBAzeeosQFIyOFOTTMyBgCAF2UI0jEGAEAtuu44MgYAwOFAdlAyBgAAzQOw2DEGAADuQK/fMQYAgHfp5AXIKLVSujIyBgDALfiZ
ATAsQK5ID14yBgAALpgGAZhQQAKBmjcyBgDAjz6jLDIGAAD6mhUEAQhszfQDPjIGAMDiETPRMQYAAJGj5UgyBgCAlBbpRAEICHPm
eQE4EIB7TkbJASAIWogAJQAMy/oPVAEgCL8z7wEgMMBDTazPMQYAgFAH1P0BCAxTPmNSASAM7Pbc3AEQCK3UpQGYEAAomiDWARBM
Eaoy1zEGAECv5PTqMQYAgDmnKDQBMGw1vKbDMQYAQJU+/LcxBgAAopv8ujEGAICZDhdMASAM/MUC5wEQbGMJJOkxBgAAHHY6LTIG
AAA5rpfQMQYAQGElz+EBCAyixlgCARgM2uvvOQEICFPB+gF4bEBTYv7xMQYAgNEyVSAyBgDAws1S/DEGAECnHdMB2BAAshOF6wEQ
KHwcGy4yBgCAxFWSBegovZLoTzIGAMBUj+shgBCASX95xAEoLMUjQ8UxBgDAiw1n1QEIDKGSeNMBCCwHYunIMQYAgM7FV8oBCCiP
N0QjMgYAANewDSHwDIAlE6MFMAgVbbQB+AzASk3lAQgQgO5CdAYBKAxAYeyyATgIaUuNAfAswOrEvgcyBgCAyfTiBUDIh1gVvDEG
AECudKAUMgYAAEWhArwxBgDAPDp9EzIGAICgIKa9MQYAQDGumy8yBgCAO1ORQdAwABazvj8yBgAAvvcxwgEwKEZAnd4xBgAA9D31
QRAMwOL+ISGAbMDfIRAoMgYAwGOwqBgyBgAA/HzsyzEGAIAEHfUBOHSAk6rV7TEGAMDW4VM8MgYAQAsR5RcyBgAAMHAJADIBiETv
z+gxBgCASOqrWTIGAIAw9qVBaAyAgtUSQagQgGNZxFcBGEgzaqk8MgYAQOBKfP4xBgDAUwpqYWgsgD8JIcAxBgBA23WrQSAMwCj0
YAWACKrEz2FYLADDjMkDMgYAwCzwxSGoDABYEV0hiAyAmQqyJaAo9nvquDEGAMBk2jkBOAwAA+7iQZgsAHlz+VQyBgCA6OmdBegI
4NX5IXAMgBlvL2EAbMBhws8nMgYAAHV8dbsxBgAAUETDFTIGAIA+e1kFQEiZNSfwMQYAgFOa6jIyBgBAGZPlATAMgOP2YgH4EEDw
dKRVARgINDsoIYgQAGxsr/kBMAiy/MBheCyATQw0vzEGAADnRXthEAyALv1hBUgowNUO8jEGAICoEMEBoAzAaEbFJWAMyEBA9QEY
KJ0foCEyBgDAER4FQcBMQAEwUFsyBgAAwNW+zTEGAACt8nhhaDxAGXmzujEGAEACX8LXMQYAFQAV0gIV2gIsFcACFRAVBhUGHBgI
QK5ID14yBgAYCEDN56ixMQYAFgAoCECuSA9eMgYAGAhAzeeosTEGABERAAAAqQHwqAMAAADAAgEIKQABAgMEBQYHCAkKCwwNDg8Q
ERITFBUWFxgZGhscHR4fICEiIyQlJicoKSorLC0uLzAxMjM0NTY3ODk6Ozw9Pj9AQUJDREVGR0hJSktMTU5PUFFSU1RVVldYWVpb
XF1eX2BhYmNkZWZnaGlqa2xtbm9wcXJzdHV2d3h5ent8fX5/gIGCg4SFhoeIiYqLjI2Oj5CRkpOUlZaXmJmam5ydnp8VBBlcNQAY
BnNjaGVtYRUIABUEJQIYBnR4bl9pZAAVDCUCGAhjdXJyZW5jeSUATBwAAAAVDhUMFQIYBmFtb3VudCUKFQQVGCxcFQQVGAAAABUE
JQIYCHR4bl90aW1lJRRMjBIcLAAAAAAAFsACGRwZTCYAHBUEGTUABhAZGAZ0eG5faWQVAhbAAhb4FxaaDibECiYIHBgI0HUAAAAA
AAAYCDF1AAAAAAAAFgAoCNB1AAAAAAAAGAgxdQAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgDAAgAAACYAHBUMGTUABhAZ
GAhjdXJyZW5jeRUCFsACFtACFtACJpoPJqIOHDYAKANVU0QYA0FFRBERABksFQQVABUCABUAFRAVAgA8FsAHGQYZJgDAAgAAACYA
HBUOGTUABhAZGAZhbW91bnQVAhbAAhboEhbMECbwHSbyEBwYBgAAAAeZPxgGAAAAAALIFgAoBgAAAAeZPxgGAAAAAALIEREAGSwV
BBUAFQIAFQAVEBUCADwpBhkmAMACAAAAJgAcFQQZNQAGEBkYCHR4bl90aW1lFQIWwAIW+BcW1BYmtDQmviEcGAhArkgPXjIGABgI
QM3nqLExBgAWACgIQK5ID14yBgAYCEDN56ixMQYAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAMACAAAAFqhFFsACJggWijgAGRwY
DEFSUk9XOnNjaGVtYRiYAy8vLy8veWdCQUFBUUFBQUFBQUFLQUF3QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFB
SUFBQUFCQUFBQUFRQUFBQzhBQUFBZUFBQUFEd0FBQUFFQUFBQVpQLy8vd0FBQVFvUUFBQUFJQUFBQUFRQUFBQUFBQUFBQ0FBQUFI
UjRibDkwYVcxbEFBQUdBQWdBQmdBR0FBQUFBQUFDQUpqLy8vOEFBQUVIRUFBQUFDQUFBQUFFQUFBQUFBQUFBQVlBQUFCaGJXOTFi
blFBQUFnQURBQUVBQWdBQ0FBQUFBd0FBQUFDQUFBQTBQLy8vd0FBQVFVUUFBQUFJQUFBQUFRQUFBQUFBQUFBQ0FBQUFHTjFjbkps
Ym1ONUFBQUFBQVFBQkFBRUFBQUFFQUFVQUFnQUJnQUhBQXdBQUFBUUFCQUFBQUFBQUFFQ0VBQUFBQ0FBQUFBRUFBQUFBQUFBQUFZ
QUFBQjBlRzVmYVdRQUFBZ0FEQUFJQUFjQUNBQUFBQUFBQUFGQUFBQUFBQUFBQUE9PQAYIHBhcnF1ZXQtY3BwLWFycm93IHZlcnNp
b24gMjUuMC4xGUwcAAAcAAAcAAAcAAAA7gMAAFBBUjE=
""",
}

_data_dir = pathlib.Path("data") / "010-asof-exchange-rate-join"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

exchange_rates = spark.read.parquet((_data_dir / "exchange_rates.parquet").resolve().as_posix())
exchange_rates.createOrReplaceTempView("exchange_rates")
transactions = spark.read.parquet((_data_dir / "transactions.parquet").resolve().as_posix())
transactions.createOrReplaceTempView("transactions")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        return passed

    return check

EXPECTED = {'column_fingerprints': {'amount': '8c6c9dd8d003689f7e543a805f7a5d999fbddac38b03cfbca1110a1a415cc60b',
                         'amount_inr': 'f7205877a71e4aa44fabbc2bc1c04d253af4d9ff9eb585a857508ac31db8b7bc',
                         'currency': '00f31e049b851c76ee9ac80f744877bf22daada37803d05c1da3de001b2314be',
                         'rate_to_inr': '5bb7c24360090878d5a51a7bc38f535ad0d885c4f451ca96e6c00a61e6c15164',
                         'txn_id': '553ca6f4afb741bed11fb1d8e5c4131ff256a42f510abdc714cc2467fa0e70a0'},
 'columns': [['amount', 'decimal(12,2)'],
             ['amount_inr', 'decimal(16,2)'],
             ['currency', 'string'],
             ['rate_to_inr', 'decimal(10,4)'],
             ['txn_id', 'bigint']],
 'fingerprint': 'e5da4f6f5339fe964d909c3dff05888a27f8b0fca810c6e96f005bc54b4bc1b2',
 'order_matters': False,
 'row_count': 160}

check = make_check(EXPECTED)

## The situation

Finance reports all transactions in INR. Exchange rates are published at irregular
times, sometimes several times a day, and each rate applies from the moment it is
published until the next one for that currency.

Every transaction must be converted with the rate that was in force when it
happened, not today's rate and not a daily average.

## Input tables

### `transactions`

One row per transaction.

| Column | Type | Description |
|---|---|---|
| `txn_id` | BIGINT | Unique. |
| `currency` | STRING | Currency code of the transaction, for example `USD` or `INR`. |
| `amount` | DECIMAL(12,2) | Amount in that currency. |
| `txn_time` | TIMESTAMP_NTZ | When the transaction happened. |

### `exchange_rates`

One row per published rate. Some rows were delivered twice and are identical in
every column. `exchange_rates` has no rows for INR.

| Column | Type | Description |
|---|---|---|
| `currency` | STRING | The foreign currency. |
| `rate_to_inr` | DECIMAL(10,4) | INR for one unit of the currency. |
| `valid_from` | TIMESTAMP_NTZ | The rate applies from this moment until the currency's next rate. |

## Requirement

Return every transaction exactly once, with the rate used and the amount in INR.

- Use the rate of the transaction's currency with the latest `valid_from` that is
  **at or before** `txn_time`. A rate published at the exact moment of the transaction
  applies to it.
- INR transactions use a rate of 1.
- If no rate for the currency had been published by `txn_time`, the transaction
  still appears, with `rate_to_inr` and `amount_inr` both NULL.
- `amount_inr` is `amount` times the rate, rounded to 2 decimals (half up, which is
  what Spark's `round` does).

## Expected output

One row per transaction.

| Column | Type | Description |
|---|---|---|
| `txn_id` | BIGINT | |
| `currency` | STRING | |
| `amount` | DECIMAL(12,2) | The original amount. |
| `rate_to_inr` | DECIMAL(10,4) | The rate used. 1.0000 for INR. NULL if no rate was known. |
| `amount_inr` | DECIMAL(16,2) | NULL if no rate was known. |

In [ ]:
exchange_rates.show(5)
transactions.show(5)

## Your answer

Use the DataFrame API. The tables are available as DataFrames: `exchange_rates`, `transactions`.

In [ ]:
result = None  # Replace None with your DataFrame.

In [ ]:
check(result)